# Prosody verifier for Shaer: main notebook

Research line on top of the reproduced Shaer baseline: a formal Arabic prosody verifier (meter, qafiya, radeef,
poem-level consistency) used to audit the baseline and to guide generation (idea 1: verifier-guided decoding).

* All code for this research lives in [`prosody-verifier/`](https://github.com/thebazshah/Shaer/tree/dev/prosody-verifier) on branch `dev`.
  Every cell is plain Python calling `prosody-verifier/pv_colab.py`, which reuses the baseline module `colab/shaer_colab.py`
  (unchanged) for the environment, progress bars, heartbeat and stall detection. Only the installation cell uses `!` commands.
* Everything this notebook writes goes to **`MyDrive/shaer-prosody/`**. Baseline results are only read, from
  `MyDrive/shaer-repro-<model>/runs/` where the baseline notebook put them.
* Every long step shows progress and a heartbeat; a step that is silent and idle for `STALL_MINUTES` is stopped with a message,
  and re-running the cell resumes.

**Runtime.** M0 needs no GPU: pick a CPU runtime (cheapest). Idea 1's generation runs need a GPU; an L4 works with smaller batches.

| Milestone | What | Runtime | Status |
|---|---|---|---|
| M0 | Headroom audit of the baseline poems and the human reference poems | CPU, minutes | in this notebook |
| M1-M2 | Prosody rule specification, verifier, gold set | CPU | added later |
| M3 | Strict re-scoring of the baselines with the verifier | CPU | added later |
| M4 | Verifier-guided decoding (idea 1) vs best-of-N at equal compute | GPU | added later |

## 1. Configuration, Google Drive, secrets

In [ ]:
import getpass
from google.colab import drive, userdata

# ---- edit these -------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/thebazshah/Shaer.git"
BRANCH = "dev"
BASE_MODEL = "Navid-AI/Yehia-7B-preview"   # the paper's model; it selects the baseline folders (shaer-repro-yehia)
STALL_MINUTES = 15     # a step that is silent AND idle for this long is stopped, with instructions to resume
# -----------------------------------------------------------------------------------------------------------

drive.mount("/content/drive")

def secret(name, prompt=None):
    try:
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass.getpass(prompt) if prompt else ""

HF_TOKEN = secret("HF_TOKEN")            # optional for M0 (the released poems and the meter classifier are public)
GITHUB_TOKEN = secret("GITHUB_TOKEN")    # only needed if the fork is private
OPENROUTER_API_KEY = secret("OPENROUTER_API_KEY")   # only for the LLM judge (later milestones)
print("HF token set:", bool(HF_TOKEN), "| OpenRouter key set:", bool(OPENROUTER_API_KEY))

## 2. Get the code from GitHub (branch `dev`)

In [ ]:
import importlib, pathlib, subprocess, sys

REPO = pathlib.Path("/content/Shaer")
git_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only", git_url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, git_url, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)   # keep no token in .git/config

for p in (REPO / "colab", REPO / "prosody-verifier"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import shaer_colab as sc
import pv_colab as pv
importlib.reload(sc); importlib.reload(pv)      # pick up the latest versions after a pull

cfg = sc.make_config(base_model=BASE_MODEL, stall_minutes=STALL_MINUTES, repo_url=REPO_URL, branch=BRANCH,
                     hf_token=HF_TOKEN, openrouter_key=OPENROUTER_API_KEY)
sc.sync_repo(cfg)                    # the repository's run folders point at the baseline's Drive folder
pvc = pv.make_config(cfg)            # this research writes to MyDrive/shaer-prosody
pv.runtime_info()

## 3. Environment (about 5 minutes on a fresh runtime)

The same isolated environment as the baseline notebook (`/content/venv`, the repository's pinned versions). Safe to run again:
when everything is installed it finishes in seconds.

In [ ]:
!pip -q install uv
!uv venv --python 3.12 --allow-existing /content/venv
!uv pip install --python /content/venv/bin/python torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/venv/bin/python -r /content/Shaer/colab/requirements-colab.txt
!uv pip check --python /content/venv/bin/python

## 4. M0 - headroom audit

Where do the baseline poems fail, and how far are they from the human poems on the same prompts?

* **Systems:** the human reference poems of the 3,481 test prompts, the released Shaer poems (E0), and every complete baseline
  run found on Drive (E1 released adapter, E2 untuned base model, ...). Runs with several samples per prompt are audited on the
  first sample (the paper's protocol).
* **Count:** exact / too short / too long / odd final hemistich, by requested length.
* **Meter:** the repository's BiLSTM classifier per bayt: share of bayts on the requested meter, poems with every bayt on it,
  poems whose bayts get more than one meter, error rate by bayt position and by meter. Scored with a batched wrapper of the
  repository's own scorer (identical results, checked on every run; much faster on a CPU).
* **Rhyme:** a written-form **heuristic** of the rawi (rhyme letter) at the end of each bayt. It is *not* the prosody verifier;
  read it next to the reference column, which shows how the same heuristic behaves on human poems.
* **Diacritics:** how much of the text carries harakat, which tells the verifier how much vowelisation it must infer.

Outputs: `MyDrive/shaer-prosody/m0/` (per-poem rows, per-system summaries, `m0_report.md`, `m0_summary.json`).

In [ ]:
pv.run_m0(pvc)

In [ ]:
pv.show_m0(pvc)

## Next milestones

Sections for M1 onwards are added to this notebook as they are built. Pull the latest `dev` (section 2) before running them.
If you save this notebook from Colab, save it under a different name (or reload it from GitHub first), so a stale copy does
not overwrite newer cells.